# TEKNOFEST 2026 — Variant Pathogenicity Classifier
## Phases 4 → 6  (Training · Feature Selection · Stacking Ensemble)

Colab **driver** for the modelling half of the pipeline. Phases 0–3 already produced the
materialised, leakage-free feature matrices and the persisted CV folds; every heavy routine
here lives in the importable `src/` package, so the notebook only orchestrates.

**Inputs (from Phase 0–3):**
- `data/processed/development_features.parquet` — `(3511, 522)` features + `Label` + `source`, **`NaN` preserved**.
- `data/processed/canary_features.parquet` — `(291, 522)`, the 10% hold-out touched **once** at the very end.
- `outputs/folds/fold_indices.json` — the 5-fold `StratifiedGroupKFold` indices (group = variant, stratify = label+source).

**Scope (stops exactly after Phase 6):**
- **Phase 4a** — default XGBoost / LightGBM baselines through the shared CV loop; per-source AUC/F1/MCC.
- **Phase 5** — **SHAP** importances + the **Null Importance Test** (≥50 label shuffles, `joblib`-parallel) + correlation pruning ⇒ the selected feature subset.
- **Phase 4b** — **Optuna** (`MedianPruner`) tuning of XGB / LGBM / CatBoost **on the selected subset only**.
- **Phase 4.6** — tier model **heads**: separate base models for **skeleton** (≥337 missing) vs **dense** (≤20 missing) rows.
- **Phase 6** — OOF collection ⇒ **Isotonic calibration** ⇒ diversity check ⇒ **rank-normalisation** ⇒ Level-1 **Logistic-Regression meta-learner** on the calibrated+ranked OOF **plus** raw meta-features (source flags + missingness tier).

> **Constraints honoured:** no `!pip install` (Colab ships everything); GPU params come **only** from `config.model_gpu_params()`; tree models receive `NaN` **natively** (no imputation); folds are **loaded**, never regenerated.

In [4]:
!pip install xgboost lightgbm catboost optuna shap

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   -- ------------------------------------- 6.3/101.7 MB 38.6 MB/s eta 0:00:03
   --------- ------------------------------ 23.6/101.7 MB 62.2 MB/s eta 0:00:02
   --------------- ------------------------ 39.6/101.7 MB 68.0 MB/s eta 0:00:01
   ------------------ --------------------- 46.9/101.7 MB 59.7 MB/s eta 0:00:01
   --------------------------- ------------ 70.8/101.7 MB 70.5 MB/s eta 0:00:01
   ------------------------------- -------- 78.9/101.7 MB 65.4 MB/s eta 0:00:01
   --------------------------------------  101.4/101.7 MB 71.2 MB/s eta 0:00:01
   --------------------------------------  101.4/101.7 MB 71.2 MB/s eta 0:00:01
   --------------------------------------  101.4/101.7 MB 71.2 MB/s eta 0:00:01
   --------------------------------------- 101.7/101.7 MB 49.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 18.9 M


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# --- Environment setup -------------------------------------------------------
# Resolve the repo root (so ./data and ./outputs paths work) and put it on
# sys.path so `from src import ...` works whether launched from ./ or ./notebooks.
import os
import sys


def find_repo_root(start: str = None) -> str:
    p = start or os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(p, "data")) and os.path.isdir(os.path.join(p, "src")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    return start or os.getcwd()


ROOT = find_repo_root()
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd

from src import config as C
from src import cv_runner as cvr
from src import feature_selection as fsel
from src import stacking as stk

C.set_global_seed(C.SEED)
C.ensure_dirs()

# GPU parameters come EXCLUSIVELY from the centralized config (constraint #3).
USE_GPU = C.detect_gpu()
GPU_PARAMS = C.model_gpu_params(USE_GPU)

print("Repo root      :", os.getcwd())
print("Python         :", sys.version.split()[0])
print("GPU available  :", USE_GPU)
print("XGB  gpu params:", GPU_PARAMS["xgboost"])
print("LGBM gpu params:", GPU_PARAMS["lightgbm"])
print("Cat  gpu params:", GPU_PARAMS["catboost"])

Repo root      : c:\Users\fikre\Desktop\Teknofest2026
Python         : 3.12.6
GPU available  : True
XGB  gpu params: {'tree_method': 'hist', 'device': 'cuda', 'random_state': 42}
LGBM gpu params: {'device_type': 'gpu', 'max_bin': 255, 'random_state': 42}
Cat  gpu params: {'task_type': 'GPU', 'devices': '0', 'random_seed': 42}


## Phase 4a — Load Matrices, Folds & Default-Model Baselines

`cv_runner.load_development_matrix()` splits `Label`/`source` off the materialised parquet and
returns `X` with **`NaN` preserved** (`float32`, no imputation). `load_folds()` reads the **exact**
`fold_indices.json` from Phase 0.4 — folds are never regenerated.

The run-knobs below control the (expensive) Phase-5/4b budgets. The plan calls for **≥50** null
shuffles and **300** Optuna trials; lower them for a quick dry-run, then restore for the final run.

`scale_pos_weight` is computed **at runtime** as `count_benign / count_pathogenic ≈ 0.37` — Pathogenic
is the *majority* (≈74%), so this correctly **down-weights** it (plan 4.2; never hardcoded).

In [6]:
# --- Load the Phase-3 matrices + the persisted folds -------------------------
X, y, source = cvr.load_development_matrix()      # NaN preserved, no Label/source in X
X_canary, y_canary, source_canary = cvr.load_canary_matrix()
folds = cvr.load_folds()                          # exact StratifiedGroupKFold indices

ALL_FEATURES = cvr.feature_columns(X)
SPW = cvr.compute_scale_pos_weight(y)             # ~0.37 (benign / pathogenic)

# ---- Run budgets (plan: >=50 shuffles, 300 trials) --------------------------
N_NULL_SHUFFLES = 50      # Phase 5.2 null importance test
N_OPTUNA_TRIALS = 200     # Phase 4b per-model search
SHAP_MAX_SAMPLES = 2000   # rows sampled per fold for SHAP
SHAP_TOP_N = 200          # keep null-survivors that are also top-N by mean|SHAP|

print(f"development X = {X.shape} | canary X = {X_canary.shape} | folds = {len(folds)}")
print(f"pos_rate = {y.mean():.4f} | scale_pos_weight = {SPW:.4f} | n_features = {len(ALL_FEATURES)}")
print("per-source rows:", pd.Series(source).value_counts().to_dict())
# Sanity: the 5 validation folds must exactly partition the development set.
val_union = sorted(sum([f["val"] for f in folds], []))
assert val_union == list(range(len(X))), "folds must partition development"
print("OK | folds partition development; NaN preserved:", bool(X.isna().any().any()))

development X = (3511, 522) | canary X = (291, 522) | folds = 5
pos_rate = 0.7448 | scale_pos_weight = 0.3426 | n_features = 522
per-source rows: {'master': 2676, 'kanser': 373, 'pah': 354, 'cftr': 108}
OK | folds partition development; NaN preserved: True


In [7]:
# --- Phase 4a: default XGBoost / LightGBM baselines on ALL features ----------
# These quick (untuned) models (a) give a reference CV score and (b) supply the
# fold models that SHAP importance (Phase 5.5) is computed from. Each learner
# factory bakes in the GPU params from config + the runtime scale_pos_weight,
# and receives NaN natively with early stopping on the val fold.
xgb_default_factory = cvr.make_xgb_learner(y=y, use_gpu=USE_GPU)
lgbm_default_factory = cvr.make_lgbm_learner(y=y, use_gpu=USE_GPU)

res_xgb_default = cvr.run_cv(xgb_default_factory, X, y, source, folds, name="xgb_default")
res_lgbm_default = cvr.run_cv(lgbm_default_factory, X, y, source, folds, name="lgbm_default")

for r in (res_xgb_default, res_lgbm_default):
    print(f"\n{r.name}: {r.summary()}")
    print("  per-source OOF AUC:", {k: round(v, 4) for k, v in r.per_source_oof_auc.items()})

c:\Users\fikre\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:751: UserWarning: [20:47:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[cv:xgb_default] fold 0: n_train=2812 val_auc=0.8486 f1=0.8822 mcc=0.5481
[cv:xgb_default] fold 1: n_train=2815 val_auc=0.8533 f1=0.8678 mcc=0.5435
[cv:xgb_default] fold 2: n_train=2804 val_auc=0.8692 f1=0.8834 mcc=0.5910
[cv:xgb_default] fold 3: n_train=2810 val_auc=0.8736 f1=0.8854 mcc=0.5941
[cv:xgb_default] fold 4: n_train=2803 val_auc=0.8667 f1=0.8827 mcc=0.5659
[cv:xgb_default] OOF AUC=0.8604 | folds 0.8623+/-0.0096 | per-source {'master': 0.8519, 'kanser': 0.9149, 'pah': 0.7982, 'cftr': 0.9472}
[cv:lgbm_default] fold 0: n_train=2812 val_auc=0.8476 f1=0.8927 mcc=0.5534
[cv:lgbm_default] fold 1: n_train=2815 val_auc=0.8526 f1=0.8727 mcc=0.5323
[cv:lgbm_default] fold 2: n_train=2804 val_auc=0.8629 f1=0.9008 mcc=0.6102
[cv:lgbm_default] fold 3: n_train=2810 val_auc=0.8706 f1=0.8988 mcc=0.5891
[cv:lgbm_default] fold 4: n_train=2803 val_auc=0.8622 f1=0.8972 mcc=0.5798
[cv:lgbm_default] OOF AUC=0.8574 | folds 0.8592+/-0.0081 | per-source {'master': 0.8487, 'kanser': 0.9156, 'pah': 0.79

## Phase 5 — Feature Selection (run **before** Optuna)

Three complementary filters aggressively prune the 522 features down to the true signals:

1. **Null Importance Test (5.2, gold standard)** — train a default GBM on the real labels, then
   on **≥50 label-shuffled** copies (`joblib`-parallel, CPU LightGBM so processes don't contend
   for the GPU). A feature survives only if its real gain importance exceeds the **95th percentile**
   of its null distribution — i.e. it does more than fit noise.
2. **SHAP importance (5.5)** — mean |SHAP| over the default-XGB fold models, each explained on its
   own validation rows (more reliable than `gain` for correlated features). Used as the final
   arbiter: we keep null-survivors that are also **top-`SHAP_TOP_N`** by SHAP.
3. **Correlation pruning (5.4)** — for `|Pearson| > 0.95` pairs, drop the one less point-biserially
   correlated with the label.

Outputs: `outputs/feature_selection/null_importance_results.csv`,
`shap_importance_ranking.csv`, `selected_features.json`, `final_feature_list.txt`.

In [8]:
# --- Phase 5.5: SHAP importance from the default-XGB fold models -------------
# (mean |SHAP| averaged across folds, each explained on its own validation rows)
shap_imp = fsel.shap_importance_from_cv(res_xgb_default, X, folds, max_samples=SHAP_MAX_SAMPLES)
print("top-15 features by mean|SHAP|:")
print(shap_imp.head(15).round(4).to_string())

c:\Users\fikre\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


top-15 features by mean|SHAP|:
EK_7_ranked                     0.4182
misspat_sim_diff                0.3318
aa_pair_te                      0.1461
EK_7                            0.1258
AL_327                          0.0987
AL_184                          0.0736
AL_311                          0.0731
abs_delta_hydrophobicity_eis    0.0678
EK_2_ranked                     0.0649
AL_319                          0.0647
AL_16                           0.0626
EK_9_ranked                     0.0605
af_log_mean                     0.0590
AL_11                           0.0562
int_tier_x_ekmean               0.0554


In [9]:
# --- Phase 5.2 + 5.4: Null Importance Test + correlation pruning -------------
# NaN-native CPU LightGBM gain importances; joblib fans the >=50 shuffles across cores.
importance_fn = fsel.make_lgbm_importance_fn(n_estimators=400, n_jobs=1)

fs = fsel.run_feature_selection(
    importance_fn, X, y,
    n_shuffles=N_NULL_SHUFFLES,
    n_jobs=-1,
    correlation_threshold=0.95,
    shap_importance=shap_imp,   # 5.5 arbiter
    shap_top_n=SHAP_TOP_N,
    save=True,
)

SELECTED = fs.selected
print(f"\nSelected {len(SELECTED)} / {len(ALL_FEATURES)} features "
      f"(null-survivors={fs.meta['n_null_survivors']}, dropped-correlated={fs.dropped_correlated}).")
print("null-importance top-10 by null_score:")
print(fs.null_importance.head(10)[["feature", "actual_importance", "null_p95", "null_score", "keep"]]
      .round(3).to_string(index=False))

[null-imp] 46/522 features beat the null 95th pct over 50 shuffles.
[corr-prune] dropped 6 of 38 (|r|>0.95); 32 remain.
[feature-selection] saved 32 features -> ./outputs/feature_selection\selected_features.json

Selected 32 / 522 features (null-survivors=38, dropped-correlated=6).
null-importance top-10 by null_score:
            feature  actual_importance  null_p95  null_score  keep
has_population_data             87.373     0.000       4.482  True
   misspat_sim_diff           2191.889   117.911       2.915  True
               EK_7           4144.812   499.287       2.115  True
              AL_16            887.750   211.831       1.429  True
               AL_1            301.578    73.065       1.407  True
             AL_215            465.307   146.373       1.152  True
         blosum_min             75.843    26.077       1.043  True
    n_missing_total            290.154   107.527       0.987  True
  int_ekmax_over_af            485.845   180.934       0.984  True
         

## Phase 4b — Optuna Tuning (`MedianPruner`) on the Selected Subset

Now that the feature set is pruned, run the expensive search **only** on `SELECTED`. Each study
maximises **mean CV AUC** across the persisted folds, with `MedianPruner(n_startup_trials=20,
n_warmup_steps=10)` killing unpromising trials early (the objective `trial.report()`s the running
mean AUC per fold). The search spaces are exactly the plan's (4.3 / 4.4 / 4.5). Per-trial estimator
count is capped for speed; the winning params are refit at full `n_estimators` (2000) with early
stopping in the next cell.

In [10]:
# --- Phase 4b: tune XGB / LGBM / CatBoost on SELECTED features ---------------
best_xgb, study_xgb = cvr.tune_model(
    "xgb", X, y, folds, SELECTED, n_trials=N_OPTUNA_TRIALS, use_gpu=USE_GPU)
best_lgbm, study_lgbm = cvr.tune_model(
    "lgbm", X, y, folds, SELECTED, n_trials=N_OPTUNA_TRIALS, use_gpu=USE_GPU)
best_cat, study_cat = cvr.tune_model(
    "catboost", X, y, folds, SELECTED, n_trials=N_OPTUNA_TRIALS, use_gpu=USE_GPU)

print("\nBest params:")
print("  XGB     :", best_xgb)
print("  LGBM    :", best_lgbm)
print("  CatBoost:", best_cat)

[tune:xgb] best CV AUC=0.8743 over 200 trials -> {'max_depth': 10, 'learning_rate': 0.08206897732353532, 'subsample': 0.5790725055000115, 'colsample_bytree': 0.9371378697162335, 'colsample_bylevel': 0.6134684339002514, 'min_child_weight': 16, 'gamma': 1.0298276910725004, 'reg_alpha': 1.824906119937466, 'reg_lambda': 0.0006882656054980603, 'max_delta_step': 8.251841080891477}
[tune:lgbm] best CV AUC=0.8742 over 200 trials -> {'num_leaves': 16, 'learning_rate': 0.08170132144556004, 'min_child_samples': 10, 'subsample': 0.5488546305808156, 'colsample_bytree': 0.8426119349725372, 'reg_alpha': 0.008547864616012292, 'reg_lambda': 5.225599102338694e-05, 'min_split_gain': 0.5131541869046352, 'path_smooth': 9.385764277212722}


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric perio

[tune:catboost] best CV AUC=0.8763 over 200 trials -> {'depth': 7, 'learning_rate': 0.024786166984277134, 'l2_leaf_reg': 8.277267528547732, 'bagging_temperature': 0.008082329127394594, 'random_strength': 5.255320736424057, 'border_count': 145}

Best params:
  XGB     : {'max_depth': 10, 'learning_rate': 0.08206897732353532, 'subsample': 0.5790725055000115, 'colsample_bytree': 0.9371378697162335, 'colsample_bylevel': 0.6134684339002514, 'min_child_weight': 16, 'gamma': 1.0298276910725004, 'reg_alpha': 1.824906119937466, 'reg_lambda': 0.0006882656054980603, 'max_delta_step': 8.251841080891477}
  LGBM    : {'num_leaves': 16, 'learning_rate': 0.08170132144556004, 'min_child_samples': 10, 'subsample': 0.5488546305808156, 'colsample_bytree': 0.8426119349725372, 'reg_alpha': 0.008547864616012292, 'reg_lambda': 5.225599102338694e-05, 'min_split_gain': 0.5131541869046352, 'path_smooth': 9.385764277212722}
  CatBoost: {'depth': 7, 'learning_rate': 0.024786166984277134, 'l2_leaf_reg': 8.277267528

## Phase 4 (tuned base models) + Phase 4.6 (missingness heads)

Train the five Level-0 base learners through the shared CV loop, each on the `SELECTED` features,
collecting leakage-free **OOF** probabilities (full development length) **and** fold-averaged
**canary** predictions (`X_extra`) for the final stacked evaluation:

- **`XGB_tuned`, `LGBM_tuned`, `CatBoost_tuned`** — the Optuna winners refit at full depth.
- **`Skeleton_head`** — trained **only** on skeleton rows (`is_skeleton_row == 1`, ≥337 missing) using
  the EK + AA + missingness-meta subset, yet predicting OOF for **every** validation row.
- **`Dense_head`** — trained **only** on dense rows (`is_dense_row == 1`, ≤20 missing) on the full subset.

Training each head on its own tier prevents the dense-row model from being distorted by skeleton-row
gradients and vice-versa, while still giving the meta-learner a dense, tier-specialised OOF column.

In [11]:
# --- Tuned base models (refit at full n_estimators=2000, early stopping) -----
xgb_tuned_factory = cvr.make_xgb_learner(best_xgb, y=y, use_gpu=USE_GPU)
lgbm_tuned_factory = cvr.make_lgbm_learner(best_lgbm, y=y, use_gpu=USE_GPU)
cat_tuned_factory = cvr.make_catboost_learner(best_cat, y=y, use_gpu=USE_GPU)

res_xgb = cvr.run_cv(xgb_tuned_factory, X, y, source, folds, name="XGB_tuned",
                     feature_cols=SELECTED, X_extra=X_canary)
res_lgbm = cvr.run_cv(lgbm_tuned_factory, X, y, source, folds, name="LGBM_tuned",
                      feature_cols=SELECTED, X_extra=X_canary)
res_cat = cvr.run_cv(cat_tuned_factory, X, y, source, folds, name="CatBoost_tuned",
                     feature_cols=SELECTED, X_extra=X_canary)

# --- Phase 4.6 missingness heads --------------------------------------------
skeleton_mask = (X["is_skeleton_row"] == 1).to_numpy()
dense_mask = (X["is_dense_row"] == 1).to_numpy()
skeleton_features = cvr.tier_feature_cols(SELECTED, "skeleton")   # EK + AA + missingness meta
dense_features = cvr.tier_feature_cols(SELECTED, "dense")         # full selected set
print(f"skeleton rows={int(skeleton_mask.sum())} (features={len(skeleton_features)}) | "
      f"dense rows={int(dense_mask.sum())} (features={len(dense_features)})")

res_skeleton = cvr.run_cv(cvr.make_xgb_learner(best_xgb, y=y, use_gpu=USE_GPU),
                          X, y, source, folds, name="Skeleton_head",
                          feature_cols=skeleton_features, train_row_mask=skeleton_mask,
                          X_extra=X_canary, verbose=False)
res_dense = cvr.run_cv(cvr.make_xgb_learner(best_xgb, y=y, use_gpu=USE_GPU),
                       X, y, source, folds, name="Dense_head",
                       feature_cols=dense_features, train_row_mask=dense_mask,
                       X_extra=X_canary, verbose=False)

base_results = [res_xgb, res_lgbm, res_cat, res_skeleton, res_dense]
for r in base_results:
    r.save(save_models=False)   # OOF + canary preds + metrics.json
    print(f"  {r.name:15s} OOF AUC={r.oof_metrics['auc']:.4f} | "
          f"per-source={ {k: round(v, 4) for k, v in r.per_source_oof_auc.items()} }")

[cv:XGB_tuned] fold 0: n_train=2812 val_auc=0.8607 f1=0.8762 mcc=0.5531
[cv:XGB_tuned] fold 1: n_train=2815 val_auc=0.8657 f1=0.8642 mcc=0.5678
[cv:XGB_tuned] fold 2: n_train=2804 val_auc=0.8832 f1=0.8774 mcc=0.6013
[cv:XGB_tuned] fold 3: n_train=2810 val_auc=0.8867 f1=0.8836 mcc=0.5976
[cv:XGB_tuned] fold 4: n_train=2803 val_auc=0.8754 f1=0.8591 mcc=0.5461
[cv:XGB_tuned] OOF AUC=0.8729 | folds 0.8743+/-0.0099 | per-source {'master': 0.8638, 'kanser': 0.9261, 'pah': 0.8272, 'cftr': 0.9563}
[cv:LGBM_tuned] fold 0: n_train=2812 val_auc=0.8599 f1=0.8764 mcc=0.5257
[cv:LGBM_tuned] fold 1: n_train=2815 val_auc=0.8684 f1=0.8583 mcc=0.5453
[cv:LGBM_tuned] fold 2: n_train=2804 val_auc=0.8741 f1=0.8701 mcc=0.5689
[cv:LGBM_tuned] fold 3: n_train=2810 val_auc=0.8928 f1=0.8913 mcc=0.6278
[cv:LGBM_tuned] fold 4: n_train=2803 val_auc=0.8759 f1=0.8715 mcc=0.5705
[cv:LGBM_tuned] OOF AUC=0.8700 | folds 0.8742+/-0.0109 | per-source {'master': 0.8615, 'kanser': 0.9275, 'pah': 0.8002, 'cftr': 0.9523}


Default metric period is 5 because AUC is/are not implemented for GPU


[cv:CatBoost_tuned] fold 0: n_train=2812 val_auc=0.8613 f1=0.8665 mcc=0.5286


Default metric period is 5 because AUC is/are not implemented for GPU


[cv:CatBoost_tuned] fold 1: n_train=2815 val_auc=0.8679 f1=0.8627 mcc=0.5570


Default metric period is 5 because AUC is/are not implemented for GPU


[cv:CatBoost_tuned] fold 2: n_train=2804 val_auc=0.8769 f1=0.8656 mcc=0.5689


Default metric period is 5 because AUC is/are not implemented for GPU


[cv:CatBoost_tuned] fold 3: n_train=2810 val_auc=0.8924 f1=0.8840 mcc=0.5935


Default metric period is 5 because AUC is/are not implemented for GPU


[cv:CatBoost_tuned] fold 4: n_train=2803 val_auc=0.8833 f1=0.8737 mcc=0.5629
[cv:CatBoost_tuned] OOF AUC=0.8757 | folds 0.8763+/-0.0110 | per-source {'master': 0.8661, 'kanser': 0.9324, 'pah': 0.8205, 'cftr': 0.9614}
skeleton rows=880 (features=12) | dense rows=687 (features=32)
  XGB_tuned       OOF AUC=0.8729 | per-source={'master': 0.8638, 'kanser': 0.9261, 'pah': 0.8272, 'cftr': 0.9563}
  LGBM_tuned      OOF AUC=0.8700 | per-source={'master': 0.8615, 'kanser': 0.9275, 'pah': 0.8002, 'cftr': 0.9523}
  CatBoost_tuned  OOF AUC=0.8757 | per-source={'master': 0.8661, 'kanser': 0.9324, 'pah': 0.8205, 'cftr': 0.9614}
  Skeleton_head   OOF AUC=0.7041 | per-source={'master': 0.6893, 'kanser': 0.7982, 'pah': 0.6878, 'cftr': 0.7045}
  Dense_head      OOF AUC=0.6632 | per-source={'master': 0.6619, 'kanser': 0.6327, 'pah': 0.5948, 'cftr': 0.8662}


## Phase 6 — Stacking Ensemble

`run_stacking` executes the plan's order end-to-end:

1. **OOF collection (6.1)** — concatenate the five base-model OOF columns (validated: equal length, no `NaN`).
2. **Isotonic calibration (6.2)** — one `IsotonicRegression(out_of_bounds='clip')` per model, fit on its
   own OOF; calibrators persisted to `outputs/models/calibrators/` for test-time reuse.
3. **Diversity check (6.3)** — pairwise Pearson correlation of calibrated OOF (want `< 0.95`).
4. **Rank-normalisation (6.3)** — `rankdata / n` per model to erase scale differences.
5. **Meta-features (6.4)** — calibrated+ranked OOF columns **plus** raw meta-features: source flags
   (`is_master/kanser/pah/cftr`) + ordinal `missingness_tier_encoded`.
6. **Meta-learner (6.4)** — Logistic Regression with nested CV over `C` (reusing the group folds),
   yielding the honest **stacked OOF** and a final model for test time.

The same fitted stack scores the **canary** base predictions — its **only** use, as the overfit
canary check before any submission.

In [12]:
# --- Phase 6: calibrate -> rank-norm -> meta-learner -------------------------
# Level-0 OOF (development) and fold-averaged canary predictions, in a fixed order.
oof_dict = {r.name: r.oof for r in base_results}
canary_dict = {r.name: r.extra_pred for r in base_results}

stack = stk.run_stacking(
    oof_dict, y, X,
    source=source,
    folds=folds,                      # reuse group folds for honest nested meta-CV
    extra_pred_dict=canary_dict,      # score the canary with the fitted stack
    X_extra=X_canary,
    save=True,
)

print("\nCalibration ECE per base model (lower = better):")
print(stack.calibration.to_string(index=False))
print("\nCalibrated-OOF correlation (diversity, want < 0.95):")
print(stack.diversity.round(3).to_string())

[stacking] base models: ['XGB_tuned', 'LGBM_tuned', 'CatBoost_tuned', 'Skeleton_head', 'Dense_head']
[stacking] max pairwise |corr| (calibrated OOF) = 0.967 (want < 0.95)
[meta] C=0.01   nested-CV AUC=0.8800
[meta] C=0.03   nested-CV AUC=0.8798
[meta] C=0.1    nested-CV AUC=0.8797
[meta] C=0.3    nested-CV AUC=0.8796
[meta] C=1.0    nested-CV AUC=0.8796
[meta] C=3.0    nested-CV AUC=0.8795
[meta] C=10.0   nested-CV AUC=0.8795
[meta] C=30.0   nested-CV AUC=0.8795
[meta] C=100.0  nested-CV AUC=0.8795
[meta] selected C=0.01 | nested-CV AUC=0.8800
[stacking] stacked OOF AUC = 0.8800 (rank-avg baseline 0.8670)
[stacking] per-source stacked AUC = {'master': 0.8703, 'kanser': 0.9358, 'pah': 0.8338, 'cftr': 0.954}
[stacking] artefacts -> ./outputs/models/stacking

Calibration ECE per base model (lower = better):
         model      ece
 Skeleton_head 0.091453
    LGBM_tuned 0.102187
CatBoost_tuned 0.120604
     XGB_tuned 0.121801
    Dense_head 0.443313

Calibrated-OOF correlation (diversity, 

In [13]:
# --- Stacked results + the one-time canary check -----------------------------
from sklearn.metrics import roc_auc_score

best_base = max(base_results, key=lambda r: r.oof_metrics["auc"])
print("Base model OOF AUC:")
for r in sorted(base_results, key=lambda r: -r.oof_metrics["auc"]):
    print(f"  {r.name:15s} {r.oof_metrics['auc']:.4f}")

print(f"\nrank-average baseline OOF AUC : {stack.info['rank_average_baseline_auc']:.4f}")
print(f"STACKED meta-learner OOF AUC  : {stack.info['stacked_oof_auc']:.4f}  "
      f"(best single base = {best_base.name} @ {best_base.oof_metrics['auc']:.4f})")
print(f"meta-learner: LogisticRegression(C={stack.info['best_C']}), "
      f"{stack.info['n_meta_features']} meta-features")
print("per-source STACKED OOF AUC    :",
      {k: round(v, 4) for k, v in stack.info.get("stacked_oof_auc_per_source", {}).items()})

# Canary: touched exactly once. A gap < ~0.02 vs stacked OOF AUC = no CV overfit.
canary_auc = roc_auc_score(y_canary, stack.extra_pred)
print(f"\nCANARY stacked AUC (one-time): {canary_auc:.4f}  "
      f"| gap vs OOF = {abs(canary_auc - stack.info['stacked_oof_auc']):.4f}")

Base model OOF AUC:
  CatBoost_tuned  0.8757
  XGB_tuned       0.8729
  LGBM_tuned      0.8700
  Skeleton_head   0.7041
  Dense_head      0.6632

rank-average baseline OOF AUC : 0.8670
STACKED meta-learner OOF AUC  : 0.8800  (best single base = CatBoost_tuned @ 0.8757)
meta-learner: LogisticRegression(C=0.01), 10 meta-features
per-source STACKED OOF AUC    : {'master': 0.8703, 'kanser': 0.9358, 'pah': 0.8338, 'cftr': 0.954}

CANARY stacked AUC (one-time): 0.8507  | gap vs OOF = 0.0293


In [14]:
# --- Artifacts summary (end of Phase 6) --------------------------------------
import os

print("Phase 4-6 complete. Persisted artifacts:\n")
expected = [
    os.path.join(C.FEATURE_SELECTION_DIR, "null_importance_results.csv"),
    os.path.join(C.FEATURE_SELECTION_DIR, "shap_importance_ranking.csv"),
    C.SELECTED_FEATURE_LIST_PATH,
    os.path.join(C.FEATURE_SELECTION_DIR, "final_feature_list.txt"),
    os.path.join(C.OOF_DIR, "XGB_tuned_oof.npy"),
    os.path.join(C.OOF_DIR, "LGBM_tuned_oof.npy"),
    os.path.join(C.OOF_DIR, "CatBoost_tuned_oof.npy"),
    os.path.join(C.OOF_DIR, "Skeleton_head_oof.npy"),
    os.path.join(C.OOF_DIR, "Dense_head_oof.npy"),
    os.path.join(C.CALIBRATORS_DIR, "XGB_tuned_isotonic.pkl"),
    os.path.join(C.STACKING_DIR, "meta_learner.pkl"),
    os.path.join(C.STACKING_DIR, "stacked_oof.npy"),
    os.path.join(C.STACKING_DIR, "stacking_metrics.json"),
]
for p in expected:
    print(f"  [{'OK ' if os.path.exists(p) else 'MISS'}] {p}")

print("\nNext (out of scope here): Phase 7 threshold optimisation (global + per-source),\n"
      "Phase 8 SHAP/calibration report, Phase 9 ablations. Stopping exactly after Phase 6.")

Phase 4-6 complete. Persisted artifacts:

  [OK ] ./outputs/feature_selection\null_importance_results.csv
  [OK ] ./outputs/feature_selection\shap_importance_ranking.csv
  [OK ] ./outputs/feature_selection\selected_features.json
  [OK ] ./outputs/feature_selection\final_feature_list.txt
  [OK ] ./outputs/oof_predictions\XGB_tuned_oof.npy
  [OK ] ./outputs/oof_predictions\LGBM_tuned_oof.npy
  [OK ] ./outputs/oof_predictions\CatBoost_tuned_oof.npy
  [OK ] ./outputs/oof_predictions\Skeleton_head_oof.npy
  [OK ] ./outputs/oof_predictions\Dense_head_oof.npy
  [OK ] ./outputs/models/calibrators\XGB_tuned_isotonic.pkl
  [OK ] ./outputs/models/stacking\meta_learner.pkl
  [OK ] ./outputs/models/stacking\stacked_oof.npy
  [OK ] ./outputs/models/stacking\stacking_metrics.json

Next (out of scope here): Phase 7 threshold optimisation (global + per-source),
Phase 8 SHAP/calibration report, Phase 9 ablations. Stopping exactly after Phase 6.
